
### Реализация Transformer на PyTorch с энкодером и декодером для обработки и генерации последовательностей. многоголовое внимание.

In [1]:
import torch.nn as nn
import torch
import math

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.q = nn.Linear(embed_size, embed_size)
        self.k = nn.Linear(embed_size, embed_size)
        self.v = nn.Linear(embed_size, embed_size)
        self.out = nn.Linear(embed_size, embed_size)

    def forward(self, q_data, k_data, v_data, mask=None):
        batch_size, seq_len, _ = q_data.size()

        Q = self.q(q_data).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        K = self.k(k_data).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        V = self.v(v_data).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

        score = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            score = score.masked_fill(mask == 0, float("-1e20"))

        attn = torch.softmax(score, dim=-1)
        context = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch_size, -1, self.embed_size)
        return self.out(context)



In [3]:
class FeedForward(nn.Module):
    def __init__(self, embed_size, ff_hidden_mult=4):
        super().__init__()
        self.fc1 = nn.Linear(embed_size, embed_size * ff_hidden_mult)
        self.fc2 = nn.Linear(embed_size * ff_hidden_mult, embed_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [4]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        attn_out = self.attention(x, x, x)
        x = self.norm1(attn_out + x)
        ff_out = self.ff(x)
        x = self.norm2(ff_out + x)
        return x

In [5]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.masked_attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)

        self.cross_attention = MultiHeadAttention(embed_size, heads)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = FeedForward(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)

    def forward(self, x, enc_out, trg_mask):
        masked_attn_out = self.masked_attention(x, x, x, mask=trg_mask)
        x = self.norm1(masked_attn_out + x)

        cross_attn_out = self.cross_attention(q_data=x, k_data=enc_out, v_data=enc_out)
        x = self.norm2(cross_attn_out + x)

        ff_out = self.ff(x)
        x = self.norm3(ff_out + x)
        return x

In [7]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    VOCAB_SIZE = 1000
    EMBED_SIZE = 128
    NUM_LAYERS = 2
    HEADS = 4
    MAX_LENGTH = 50
    BATCH_SIZE = 8

    model = Transformer(VOCAB_SIZE, EMBED_SIZE, NUM_LAYERS, HEADS, MAX_LENGTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003)
    criterion = nn.CrossEntropyLoss()

    src_data = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, 10)).to(device)
    trg_data = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, 12)).to(device)

    model.train()
    optimizer.zero_grad()

    output = model(src_data, trg_data)

    output_reshaped = output.view(-1, VOCAB_SIZE)
    targets_reshaped = trg_data.view(-1)

    loss = criterion(output_reshaped, targets_reshaped)
    loss.backward()
    optimizer.step()

    print(f"Форма выхода Трансформера: {output.shape} -> [Батч, Длина_целевого_текста, Размер_словаря]")
    print(f"Значение функции потерь (Loss): {loss.item():.4f}")

    first_word_logits = output[0, 0, :]
    probabilities = torch.softmax(first_word_logits, dim=-1)
    predicted_word_index = torch.argmax(probabilities).item()
    print(f"Максимальная вероятность у слова с индексом: {predicted_word_index}")

Форма выхода Трансформера: torch.Size([8, 12, 1000]) -> [Батч, Длина_целевого_текста, Размер_словаря]
Значение функции потерь (Loss): 7.0352
Максимальная вероятность у слова с индексом: 240
